In [1]:
import sys
from pathlib import Path
project_root = Path.cwd().parent
sys.path.append(str(project_root))

%load_ext autoreload
%autoreload 2

from src.reserving.data import build_reserving_dataset

df_full, observed, future = build_reserving_dataset()

print("Total lignes :", len(df_full))
print("Observées :", len(observed))
print("Futures :", len(future))
print()
print("Nb compagnies :", df_full["GRCODE"].nunique())
print()

# Vérification de cohérence : les incréments doivent sommer au cumul final
check = df_full[df_full["DevelopmentLag"] == 10].copy()
check["sum_incr"] = df_full.groupby(["GRCODE", "AccidentYear"])["IncrementalPaid"].transform("sum")
# Ne fonctionne que pour AccidentYear=1998 (seule année avec tous les 10 lags dans le triangle complet)
sample = df_full[(df_full["GRCODE"] == df_full["GRCODE"].iloc[0]) & (df_full["AccidentYear"] == 1998)]
print(sample[["DevelopmentLag", "CumPaidLoss", "IncrementalPaid"]])

Total lignes : 13250
Observées : 7513
Futures : 5737

Nb compagnies : 143

Empty DataFrame
Columns: [DevelopmentLag, CumPaidLoss, IncrementalPaid]
Index: []


In [2]:
from src.reserving.data import build_triangle_for_chainladder
import chainladder as cl

# Identifier le GRCODE de State Farm
state_farm_grcode = df_full[df_full["GRNAME"] == "State Farm Mut Grp"]["GRCODE"].iloc[0]
print("GRCODE State Farm :", state_farm_grcode)

triangle_sf = build_triangle_for_chainladder(observed, grcode=state_farm_grcode)
print(triangle_sf)

# Ajustement du modèle de Mack
mack_model = cl.MackChainladder()
mack_model.fit(triangle_sf["CumPaidLoss"])

print("\nRéserves estimées (IBNR) :")
print(mack_model.ibnr_)

print("\nErreur standard des réserves :")
print(mack_model.total_mack_std_err_)

GRCODE State Farm : 1767
                         Triangle Summary
Valuation:                        2007-12
Grain:                               OYDY
Shape:                     (1, 2, 10, 10)
Index:                            [Total]
Columns:    [CumPaidLoss, IncurredLosses]

Réserves estimées (IBNR) :
              2261
1998           NaN
1999  1.724004e+04
2000  4.674008e+04
2001  1.066184e+05
2002  2.335985e+05
2003  4.420639e+05
2004  8.667519e+05
2005  1.670833e+06
2006  3.095520e+06
2007  6.643130e+06

Erreur standard des réserves :
columns     CumPaidLoss
(Total,)  324623.022121


In [3]:
# Vérification : l'année 1998 a-t-elle bien tous ses développements dans le triangle observé ?
check_1998 = observed[(observed["GRCODE"] == state_farm_grcode) & (observed["AccidentYear"] == 1998)]
print("Nombre de DevelopmentLag observés pour 1998 :", check_1998["DevelopmentLag"].nunique())

Nombre de DevelopmentLag observés pour 1998 : 0


In [4]:
total_ibnr = mack_model.ibnr_.sum()
print("Réserve IBNR totale :", total_ibnr)

total_std_err = mack_model.total_mack_std_err_.values[0][0] if hasattr(mack_model.total_mack_std_err_, 'values') else mack_model.total_mack_std_err_
print("Erreur standard totale :", total_std_err)

# Coefficient de variation -- une mesure standard en actuariat pour juger la volatilité relative des réserves
cv = total_std_err / total_ibnr
print(f"Coefficient de variation (CV) : {cv:.2%}")

Réserve IBNR totale : 13122495.993963104
Erreur standard totale : 324623.0221210446
Coefficient de variation (CV) : 2.47%


In [5]:
# Inspecter la vraie structure de l'objet retourné par le package
print(type(mack_model.total_mack_std_err_))
print(mack_model.total_mack_std_err_)
print()

# Est-ce qu'il y a une méthode/attribut plus explicite pour l'erreur standard PAR année ?
print(mack_model.mack_std_err_)

<class 'pandas.core.frame.DataFrame'>
columns     CumPaidLoss
(Total,)  324623.022121

      12             24             36             48             60             72             84             96             108            120            9999
1998   NaN            NaN            NaN            NaN            NaN            NaN            NaN            NaN            NaN            NaN            NaN
1999   NaN            NaN            NaN            NaN            NaN            NaN            NaN            NaN            NaN     990.640425     990.640425
2000   NaN            NaN            NaN            NaN            NaN            NaN            NaN            NaN    4624.456670    4748.226089    4748.226089
2001   NaN            NaN            NaN            NaN            NaN            NaN            NaN    1718.641062    5177.803337    5303.745392    5303.745392
2002   NaN            NaN            NaN            NaN            NaN            NaN    6727.956915    6992

In [6]:
import numpy as np

# Erreurs standard ultimes par année (dernière colonne du tableau)
std_err_by_year = mack_model.mack_std_err_.iloc[:, -1]

ibnr_by_year = mack_model.ibnr_.iloc[:, 0]  # rappel de vos résultats précédents

cv_by_year = std_err_by_year / ibnr_by_year
print(cv_by_year)

      12        24        36        48        60        72        84        96        108       120       9999
1998   NaN       NaN       NaN       NaN       NaN       NaN       NaN       NaN       NaN       NaN       NaN
1999   NaN       NaN       NaN       NaN       NaN       NaN       NaN       NaN       NaN  0.057462  0.057462
2000   NaN       NaN       NaN       NaN       NaN       NaN       NaN       NaN  0.098940  0.101588  0.101588
2001   NaN       NaN       NaN       NaN       NaN       NaN       NaN  0.016120  0.048564  0.049745  0.049745
2002   NaN       NaN       NaN       NaN       NaN       NaN  0.028801  0.029935  0.037113  0.037508  0.037508
2003   NaN       NaN       NaN       NaN       NaN  0.018763  0.023969  0.024396  0.026853  0.027016  0.027016
2004   NaN       NaN       NaN       NaN  0.022800  0.025055  0.026336  0.026531  0.027165  0.027240  0.027240
2005   NaN       NaN       NaN  0.032595  0.035912  0.036917  0.037463  0.037653  0.037857  0.037926  0.037926
2

In [7]:
import pandas as pd
# Vraie charge future réalisée pour State Farm (vérité terrain)
future_sf = future[future["GRCODE"] == state_farm_grcode]

# Pour chaque AccidentYear, le vrai montant payé au dernier développement (lag=10) 
# moins ce qui était déjà payé à la date d'évaluation (2007)
true_ultimate = future_sf[future_sf["DevelopmentLag"] == 10].set_index("AccidentYear")["CumPaidLoss"]
paid_at_eval = observed[observed["GRCODE"] == state_farm_grcode].sort_values("DevelopmentLag").groupby("AccidentYear")["CumPaidLoss"].last()

true_ibnr = true_ultimate - paid_at_eval
print("Vraie réserve (réalisée) par année :")
print(true_ibnr)

print("\nComparaison avec l'estimation Mack :")
ibnr_mack_series = mack_model.ibnr_.to_frame().iloc[:, 0]

# Convertir l'index datetime en simple année (entier), pour matcher true_ibnr
ibnr_mack_series.index = ibnr_mack_series.index.year

comparison = pd.DataFrame({
    "IBNR_Mack": ibnr_mack_series,
    "IBNR_reel": true_ibnr
})
comparison["ecart"] = comparison["IBNR_Mack"] - comparison["IBNR_reel"]
comparison["ratio"] = comparison["IBNR_Mack"] / comparison["IBNR_reel"]
print(comparison)

Vraie réserve (réalisée) par année :
AccidentYear
1998-01-01          NaN
1999-01-01      22378.0
2000-01-01      52437.0
2001-01-01     115830.0
2002-01-01     247835.0
2003-01-01     496611.0
2004-01-01     919882.0
2005-01-01    1739231.0
2006-01-01    3167835.0
2007-01-01    6696665.0
Name: CumPaidLoss, dtype: float64

Comparaison avec l'estimation Mack :
                        IBNR_Mack  IBNR_reel  ecart  ratio
1998                          NaN        NaN    NaN    NaN
1999                 1.724004e+04        NaN    NaN    NaN
2000                 4.674008e+04        NaN    NaN    NaN
2001                 1.066184e+05        NaN    NaN    NaN
2002                 2.335985e+05        NaN    NaN    NaN
2003                 4.420639e+05        NaN    NaN    NaN
2004                 8.667519e+05        NaN    NaN    NaN
2005                 1.670833e+06        NaN    NaN    NaN
2006                 3.095520e+06        NaN    NaN    NaN
2007                 6.643130e+06        NaN    

In [8]:
std_err_df = mack_model.mack_std_err_.to_frame()
print(std_err_df.columns)
print(std_err_df.head())

Index([12, 24, 36, 48, 60, 72, 84, 96, 108, 120, 9999], dtype='int64')
            12    24    36    48    60    72           84           96    \
1998-01-01   NaN   NaN   NaN   NaN   NaN   NaN          NaN          NaN   
1999-01-01   NaN   NaN   NaN   NaN   NaN   NaN          NaN          NaN   
2000-01-01   NaN   NaN   NaN   NaN   NaN   NaN          NaN          NaN   
2001-01-01   NaN   NaN   NaN   NaN   NaN   NaN          NaN  1718.641062   
2002-01-01   NaN   NaN   NaN   NaN   NaN   NaN  6727.956915  6992.696699   

                   108          120          9999  
1998-01-01          NaN          NaN          NaN  
1999-01-01          NaN   990.640425   990.640425  
2000-01-01  4624.456670  4748.226089  4748.226089  
2001-01-01  5177.803337  5303.745392  5303.745392  
2002-01-01  8669.515759  8761.828576  8761.828576  


In [9]:
std_err_by_year = std_err_df[9999]  # ou le nom exact de colonne vu ci-dessus
std_err_by_year.index = std_err_by_year.index.year  # aligner sur année entière comme avant

comparison["std_err"] = std_err_by_year
comparison["lower_90"] = comparison["IBNR_Mack"] - 1.645 * comparison["std_err"]
comparison["upper_90"] = comparison["IBNR_Mack"] + 1.645 * comparison["std_err"]
comparison["covered_90"] = (comparison["IBNR_reel"] >= comparison["lower_90"]) & (comparison["IBNR_reel"] <= comparison["upper_90"])

print(comparison[["IBNR_Mack", "IBNR_reel", "std_err", "lower_90", "upper_90", "covered_90"]])
print()
print("Taux de couverture (hors 1998) :", comparison["covered_90"].iloc[1:].mean())

                        IBNR_Mack  IBNR_reel        std_err      lower_90  \
1998                          NaN        NaN            NaN           NaN   
1999                 1.724004e+04        NaN     990.640425  1.561044e+04   
2000                 4.674008e+04        NaN    4748.226089  3.892925e+04   
2001                 1.066184e+05        NaN    5303.745392  9.789372e+04   
2002                 2.335985e+05        NaN    8761.828576  2.191853e+05   
2003                 4.420639e+05        NaN   11942.598354  4.224183e+05   
2004                 8.667519e+05        NaN   23609.934023  8.279136e+05   
2005                 1.670833e+06        NaN   63367.609122  1.566593e+06   
2006                 3.095520e+06        NaN  134305.147781  2.874588e+06   
2007                 6.643130e+06        NaN  273304.080377  6.193545e+06   
1998-01-01 00:00:00           NaN        NaN            NaN           NaN   
1999-01-01 00:00:00           NaN    22378.0            NaN           NaN   

In [10]:
from src.reserving.models import evaluate_mack_coverage
from src.reserving.data import get_all_grcodes

all_grcodes = get_all_grcodes(df_full)
print("Nombre de compagnies :", len(all_grcodes))

all_results = []
failed = []

for grcode in all_grcodes:
    res = evaluate_mack_coverage(observed, future, grcode)
    if res is not None:
        all_results.append(res)
    else:
        failed.append(grcode)

print(f"Compagnies évaluées avec succès : {len(all_results)}")
print(f"Compagnies échouées (triangle dégénéré/trop petit) : {len(failed)}")

full_results = pd.concat(all_results)
print(f"\nNombre total d'observations (compagnie x année) : {len(full_results)}")

Nombre de compagnies : 143


d:\Téléchargements\projet_actuariat\src\reserving\models.py:54: RuntimeWarning: '<' not supported between instances of 'Timestamp' and 'int', sort order is undefined for incomparable objects.
  results = pd.DataFrame({
d:\Téléchargements\projet_actuariat\src\reserving\models.py:54: RuntimeWarning: '<' not supported between instances of 'Timestamp' and 'int', sort order is undefined for incomparable objects.
  results = pd.DataFrame({
d:\Téléchargements\projet_actuariat\src\reserving\models.py:54: RuntimeWarning: '<' not supported between instances of 'Timestamp' and 'int', sort order is undefined for incomparable objects.
  results = pd.DataFrame({
d:\Téléchargements\projet_actuariat\src\reserving\models.py:54: RuntimeWarning: '<' not supported between instances of 'Timestamp' and 'int', sort order is undefined for incomparable objects.
  results = pd.DataFrame({
d:\Téléchargements\projet_actuariat\src\reserving\models.py:54: RuntimeWarning: '<' not supported between instances of 'Time

Compagnies évaluées avec succès : 0
Compagnies échouées (triangle dégénéré/trop petit) : 143


ValueError: No objects to concatenate

In [ ]:
print("Taux de couverture global (90% visé) :", full_results["covered_90"].mean())
print()

# Répartition : sur-couverture (réel < borne basse) vs sous-couverture (réel > borne haute)
under = (full_results["ibnr_reel"] > full_results["upper_90"]).mean()
over = (full_results["ibnr_reel"] < full_results["lower_90"]).mean()
print(f"Proportion où le réel dépasse la borne haute (sous-estimation Mack) : {under:.2%}")
print(f"Proportion où le réel est sous la borne basse (sur-estimation Mack) : {over:.2%}")

# Ratio moyen prédit/réel, pour quantifier le biais global
full_results["ratio"] = full_results["ibnr_mack"] / full_results["ibnr_reel"]
print(f"\nRatio moyen IBNR_Mack / IBNR_réel : {full_results['ratio'].mean():.4f}")
print(f"Ratio médian : {full_results['ratio'].median():.4f}")

NameError: name 'full_results' is not defined

In [ ]:
# Identifier les cas problématiques
print("Nombre de ratios infinis ou NaN :", (~np.isfinite(full_results["ratio"])).sum())

# Ratio moyen recalculé en excluant les valeurs non finies
clean_ratios = full_results["ratio"][np.isfinite(full_results["ratio"])]
print(f"Ratio moyen (nettoyé) : {clean_ratios.mean():.4f}")
print(f"Ratio moyen tronqué (5e-95e percentile) : {clean_ratios.clip(clean_ratios.quantile(0.05), clean_ratios.quantile(0.95)).mean():.4f}")

NameError: name 'full_results' is not defined

In [ ]:
import torch
print(torch.__version__)
print("GPU disponible :", torch.cuda.is_available())

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device utilisé :", device)

2.13.0+cpu
GPU disponible : False
Device utilisé : cpu


In [ ]:
from src.reserving.data import build_sequences
from src.reserving.deep_triangle import (
    TriangleSeqDataset, DeepTriangleGRU, train_deep_triangle, predict_future_increments
)

X, M, keys, premiums = build_sequences(df_full)
print("Nombre de séquences (compagnie x année) :", len(X))

dataset = TriangleSeqDataset(X, M)

model_dt = DeepTriangleGRU(hidden_dim=16)
optimizer = torch.optim.Adam(model_dt.parameters(), lr=1e-3)

model_dt = train_deep_triangle(model_dt, dataset, n_epochs=200, optimizer=optimizer, device=device)

Compagnies exclues (prime < 100,000$) : 135
Nombre de séquences (compagnie x année) : 79
Epoch 1/200 — loss: 0.029935
Epoch 21/200 — loss: 0.006211
Epoch 41/200 — loss: 0.004447
Epoch 61/200 — loss: 0.002657
Epoch 81/200 — loss: 0.001646
Epoch 101/200 — loss: 0.001100
Epoch 121/200 — loss: 0.000754
Epoch 141/200 — loss: 0.000501
Epoch 161/200 — loss: 0.000537
Epoch 181/200 — loss: 0.000368
Epoch 200/200 — loss: 0.000331


In [ ]:
results_dt = []

for i, (grcode, ay) in enumerate(keys):
    obs_len = int(M[i].sum())
    if obs_len == 0 or obs_len == 10:
        continue  # rien à prédire

    obs_seq = X[i, :obs_len]
    future_preds_scaled = predict_future_increments(model_dt, obs_seq, device)

    predicted_ibnr = future_preds_scaled.sum() * premiums[i]

    true_seq_future = df_full[
        (df_full["GRCODE"] == grcode) & (df_full["AccidentYear"] == ay) & (df_full["DevelopmentLag"] > obs_len)
    ]
    true_ibnr_val = true_seq_future["IncrementalPaid"].sum()

    results_dt.append({"grcode": grcode, "accident_year": ay, "ibnr_dt": predicted_ibnr, "ibnr_reel": true_ibnr_val})

results_dt = pd.DataFrame(results_dt)
results_dt["ratio"] = results_dt["ibnr_dt"] / results_dt["ibnr_reel"]

print("Nombre d'observations :", len(results_dt))
print("Ratio médian IBNR_DeepTriangle / IBNR_réel :", results_dt["ratio"].replace([np.inf,-np.inf], np.nan).dropna().median())
print("Ratio médian Mack (rappel) : 1.0635")

# Erreur absolue moyenne en dollars, comparable directement
print("\nMAE Deep Triangle :", (results_dt["ibnr_dt"] - results_dt["ibnr_reel"]).abs().mean())

Nombre d'observations : 71
Ratio médian IBNR_DeepTriangle / IBNR_réel : 1.980671278255683
Ratio médian Mack (rappel) : 1.0635

MAE Deep Triangle : 63005.94908281762


In [ ]:
# Vérifier la distribution des ScaledIncr utilisés à l'entraînement
scaled_values = X[M].flatten()  # toutes les valeurs observées, aplaties
scaled_values = scaled_values[scaled_values != 0]  # enlever les zéros structurels

print("Describe des ScaledIncr observés :")
print(pd.Series(scaled_values).describe())
print()
print("Skewness :", pd.Series(scaled_values).skew())

Describe des ScaledIncr observés :
count    439.000000
mean       0.119270
std        0.124362
min       -0.000056
25%        0.014895
50%        0.075120
75%        0.203858
max        0.461198
dtype: float64

Skewness : 1.039892824049621


In [ ]:
# Prendre la séquence de State Farm 1998 (entièrement observée, donc on peut comparer)
idx_sf_1998 = [i for i, (g, ay) in enumerate(keys) if g == state_farm_grcode and ay == 1998][0]
print("Séquence observée (State Farm 1998, complète) :")
print(X[idx_sf_1998])

# Tester la prédiction sur une séquence tronquée à 5 points, comparée aux 5 vrais points suivants
obs_5 = X[idx_sf_1998, :5]
true_next_5 = X[idx_sf_1998, 5:]

pred_5 = predict_future_increments(model_dt, obs_5, device)
print("\nVrais 5 prochains points  :", true_next_5)
print("Prédits 5 prochains points :", pred_5)

IndexError: list index out of range

In [ ]:
# Identifier les compagnies avec des primes très faibles ou nulles, source probable des valeurs aberrantes
low_premium = df_full[df_full["EarnedPremNet"] < 1000]
print("Nombre de lignes avec EarnedPremNet < 1000 :", len(low_premium))
print("Compagnies concernées :", low_premium["GRCODE"].nunique())
print()

# Les valeurs extrêmes de ScaledIncr viennent-elles bien de ces compagnies ?
extreme = df_full[(df_full["IncrementalPaid"] / df_full["EarnedPremNet"].clip(lower=1)).abs() > 5]
print("Nombre de lignes avec |ScaledIncr| > 5 :", len(extreme))
print(extreme[["GRCODE", "GRNAME", "AccidentYear", "DevelopmentLag", "IncrementalPaid", "EarnedPremNet"]].head(10))

Nombre de lignes avec EarnedPremNet < 1000 : 3110
Compagnies concernées : 51

Nombre de lignes avec |ScaledIncr| > 5 : 24
      GRCODE                         GRNAME  AccidentYear  DevelopmentLag  \
3308   10019  Overseas Partners Us Reins Co          2005               9   
3309   10019  Overseas Partners Us Reins Co          2005              10   
3317   10019  Overseas Partners Us Reins Co          2006               8   
3318   10019  Overseas Partners Us Reins Co          2006               9   
3319   10019  Overseas Partners Us Reins Co          2006              10   
3326   10019  Overseas Partners Us Reins Co          2007               7   
3327   10019  Overseas Partners Us Reins Co          2007               8   
3328   10019  Overseas Partners Us Reins Co          2007               9   
3329   10019  Overseas Partners Us Reins Co          2007              10   
4181   11150              First Amer Ins Co          2002               2   

      IncrementalPaid  EarnedP

In [ ]:
from src.reserving.data import build_sequences

X, M, keys, premiums = build_sequences(df_full, min_premium=100_000)

print("Nombre de séquences après filtrage :", len(X))

scaled_values = X[M].flatten()
scaled_values = scaled_values[scaled_values != 0]
print("\nDescribe des ScaledIncr après filtrage :")
print(pd.Series(scaled_values).describe())
print("Skewness :", pd.Series(scaled_values).skew())

Compagnies exclues (prime < 100,000$) : 135
Nombre de séquences après filtrage : 79

Describe des ScaledIncr après filtrage :
count    439.000000
mean       0.119270
std        0.124362
min       -0.000056
25%        0.014895
50%        0.075120
75%        0.203858
max        0.461198
dtype: float64
Skewness : 1.039892824049621


In [ ]:
# Distribution de la prime minimale par compagnie, pour choisir un seuil qui garde plus de volume
min_prem_by_company = df_full.groupby("GRCODE")["EarnedPremNet"].min()
print(min_prem_by_company.describe())
print()
print("Nombre de compagnies par tranche de seuil :")
for threshold in [0, 1000, 5000, 10000, 20000, 50000, 100000]:
    n_valid = (min_prem_by_company >= threshold).sum()
    print(f"  seuil >= {threshold:>7,}$ : {n_valid} compagnies conservées")

count    1.430000e+02
mean     1.248652e+05
std      1.132382e+06
min     -5.390000e+02
25%      7.750000e+01
50%      2.558000e+03
75%      1.158200e+04
max      1.340184e+07
Name: EarnedPremNet, dtype: float64

Nombre de compagnies par tranche de seuil :
  seuil >=       0$ : 139 compagnies conservées
  seuil >=   1,000$ : 92 compagnies conservées
  seuil >=   5,000$ : 57 compagnies conservées
  seuil >=  10,000$ : 40 compagnies conservées
  seuil >=  20,000$ : 21 compagnies conservées
  seuil >=  50,000$ : 13 compagnies conservées
  seuil >= 100,000$ : 8 compagnies conservées


In [ ]:
X, M, keys, premiums = build_sequences(df_full, min_premium=1000)

print("Nombre de séquences après filtrage (seuil 1000$) :", len(X))

scaled_values = X[M].flatten()
scaled_values = scaled_values[scaled_values != 0]
print("\nDescribe des ScaledIncr :")
print(pd.Series(scaled_values).describe())
print("Skewness :", pd.Series(scaled_values).skew())

Compagnies exclues (prime < 1,000$) : 51
Nombre de séquences après filtrage (seuil 1000$) : 866

Describe des ScaledIncr :
count    4643.000000
mean        0.113650
std         0.121374
min        -0.373035
25%         0.013687
50%         0.067490
75%         0.194643
max         0.872695
dtype: float64
Skewness : 1.1694003870620215


In [ ]:
from src.reserving.deep_triangle import TriangleSeqDataset, DeepTriangleGRU, train_deep_triangle

dataset = TriangleSeqDataset(X, M)

model_dt = DeepTriangleGRU(hidden_dim=16)
optimizer = torch.optim.Adam(model_dt.parameters(), lr=1e-3)

model_dt = train_deep_triangle(model_dt, dataset, n_epochs=200, optimizer=optimizer, device=device)

Epoch 1/200 — loss: 0.017051
Epoch 21/200 — loss: 0.001827
Epoch 41/200 — loss: 0.001819
Epoch 61/200 — loss: 0.001770
Epoch 81/200 — loss: 0.001792
Epoch 101/200 — loss: 0.001785
Epoch 121/200 — loss: 0.001770
Epoch 141/200 — loss: 0.001775
Epoch 161/200 — loss: 0.001786
Epoch 181/200 — loss: 0.001774
Epoch 200/200 — loss: 0.001756


In [ ]:
results_dt = []

for i, (grcode, ay) in enumerate(keys):
    obs_len = int(M[i].sum())
    if obs_len == 0 or obs_len == 10:
        continue

    obs_seq = X[i, :obs_len]
    future_preds_scaled = predict_future_increments(model_dt, obs_seq, device)

    predicted_ibnr = future_preds_scaled.sum() * premiums[i]

    true_seq_future = df_full[
        (df_full["GRCODE"] == grcode) & (df_full["AccidentYear"] == ay) & (df_full["DevelopmentLag"] > obs_len)
    ]
    true_ibnr_val = true_seq_future["IncrementalPaid"].sum()

    results_dt.append({"grcode": grcode, "accident_year": ay, "ibnr_dt": predicted_ibnr, "ibnr_reel": true_ibnr_val})

results_dt = pd.DataFrame(results_dt)
results_dt["ratio"] = results_dt["ibnr_dt"] / results_dt["ibnr_reel"]

print("Nombre d'observations :", len(results_dt))
clean_ratio = results_dt["ratio"].replace([np.inf, -np.inf], np.nan).dropna()
print("Ratio médian IBNR_DeepTriangle / IBNR_réel :", clean_ratio.median())
print("Ratio moyen tronqué (5-95%) :", clean_ratio.clip(clean_ratio.quantile(0.05), clean_ratio.quantile(0.95)).mean())
print("\nRappel Mack : médian 1.0635, moyen tronqué 1.2658")

print("\nMAE Deep Triangle :", (results_dt["ibnr_dt"] - results_dt["ibnr_reel"]).abs().median())

Nombre d'observations : 774
Ratio médian IBNR_DeepTriangle / IBNR_réel : 1.2875845030700088
Ratio moyen tronqué (5-95%) : 1.983627780095855

Rappel Mack : médian 1.0635, moyen tronqué 1.2658

MAE Deep Triangle : 163.33322386443615


In [ ]:
results_dt["obs_len"] = results_dt["accident_year"].apply(
    lambda ay: M[[i for i, k in enumerate(keys) if k[1] == ay][0]].sum() if any(k[1] == ay for k in keys) else None
)

# Plus simple : recalculer obs_len directement dans la boucle précédente -- ajoutez-le à results_dt.append(...)
# Si vous n'avez pas cette colonne, relancez la boucle d'évaluation en ajoutant "obs_len": obs_len au dictionnaire

print(results_dt.groupby("obs_len")["ratio"].median())
print()
print(results_dt.groupby("obs_len")["ratio"].count())

obs_len
1    1.206605
2    1.296604
3    1.453885
4    1.419933
5    1.764667
6    1.586727
7    1.100506
8    1.686196
9    2.877157
Name: ratio, dtype: float64

obs_len
1    81
2    83
3    83
4    85
5    85
6    86
7    89
8    88
9    88
Name: ratio, dtype: int64


In [ ]:
results_dt = []

for i, (grcode, ay) in enumerate(keys):
    obs_len = int(M[i].sum())
    if obs_len == 0 or obs_len == 10:
        continue

    obs_seq = X[i, :obs_len]
    future_preds_scaled = predict_future_increments(model_dt, obs_seq, device)
    predicted_ibnr = future_preds_scaled.sum() * premiums[i]

    true_seq_future = df_full[
        (df_full["GRCODE"] == grcode) & (df_full["AccidentYear"] == ay) & (df_full["DevelopmentLag"] > obs_len)
    ]
    true_ibnr_val = true_seq_future["IncrementalPaid"].sum()

    results_dt.append({
        "grcode": grcode, "accident_year": ay, "obs_len": obs_len,
        "ibnr_dt": predicted_ibnr, "ibnr_reel": true_ibnr_val
    })

results_dt = pd.DataFrame(results_dt)
results_dt["ratio"] = results_dt["ibnr_dt"] / results_dt["ibnr_reel"]

clean = results_dt.replace([np.inf, -np.inf], np.nan).dropna(subset=["ratio"])
print(clean.groupby("obs_len")["ratio"].median())
print()
print(clean.groupby("obs_len")["ratio"].count())

obs_len
1    1.206605
2    1.296604
3    1.453885
4    1.419933
5    1.703200
6    1.554003
7    0.843297
8    0.855782
9    0.260057
Name: ratio, dtype: float64

obs_len
1    81
2    83
3    83
4    85
5    82
6    84
7    78
8    65
9    54
Name: ratio, dtype: int64


In [ ]:
model_dt = DeepTriangleGRU(hidden_dim=16)
optimizer = torch.optim.Adam(model_dt.parameters(), lr=1e-3)
model_dt = train_deep_triangle(model_dt, dataset, n_epochs=200, optimizer=optimizer, device=device)

Epoch 1/200 — loss: 0.007617
Epoch 21/200 — loss: 0.002009
Epoch 41/200 — loss: 0.001853
Epoch 61/200 — loss: 0.001846
Epoch 81/200 — loss: 0.001788
Epoch 101/200 — loss: 0.001773
Epoch 121/200 — loss: 0.001791
Epoch 141/200 — loss: 0.001780
Epoch 161/200 — loss: 0.001806
Epoch 181/200 — loss: 0.001775
Epoch 200/200 — loss: 0.001810


In [ ]:
results_dt = []

for i, (grcode, ay) in enumerate(keys):
    obs_len = int(M[i].sum())
    if obs_len == 0 or obs_len == 10:
        continue

    obs_seq = X[i, :obs_len]
    future_preds_scaled = predict_future_increments(model_dt, obs_seq, device)
    predicted_ibnr = future_preds_scaled.sum() * premiums[i]

    true_seq_future = df_full[
        (df_full["GRCODE"] == grcode) & (df_full["AccidentYear"] == ay) & (df_full["DevelopmentLag"] > obs_len)
    ]
    true_ibnr_val = true_seq_future["IncrementalPaid"].sum()

    results_dt.append({
        "grcode": grcode, "accident_year": ay, "obs_len": obs_len,
        "ibnr_dt": predicted_ibnr, "ibnr_reel": true_ibnr_val
    })

results_dt = pd.DataFrame(results_dt)
results_dt["ratio"] = results_dt["ibnr_dt"] / results_dt["ibnr_reel"]

clean = results_dt.replace([np.inf, -np.inf], np.nan).dropna(subset=["ratio"])

print("Ratio médian global :", clean["ratio"].median())
print("Ratio moyen tronqué (5-95%) :", clean["ratio"].clip(clean["ratio"].quantile(0.05), clean["ratio"].quantile(0.95)).mean())
print()
print("Par obs_len :")
print(clean.groupby("obs_len")["ratio"].median())
print()
print("Rappel : sans contrainte positive, médian global était 1.41")
print("Rappel Mack : médian 1.0635")

Ratio médian global : 2.650846399302068
Ratio moyen tronqué (5-95%) : 5.700964740322653

Par obs_len :
obs_len
1    1.396354
2    1.723842
3    2.356625
4    2.782617
5    4.500973
6    6.650033
7    4.999322
8    8.694195
9    3.109007
Name: ratio, dtype: float64

Rappel : sans contrainte positive, médian global était 1.41
Rappel Mack : médian 1.0635


In [ ]:
print("ibnr_dt describe :")
print(results_dt["ibnr_dt"].describe())
print()
print("ibnr_reel describe :")
print(results_dt["ibnr_reel"].describe())
print()
print("Nombre de ibnr_reel <= 0 :", (results_dt["ibnr_reel"] <= 0).sum())
print("Nombre de ibnr_dt <= 0 :", (results_dt["ibnr_dt"] <= 0).sum())

ibnr_dt describe :
count    7.740000e+02
mean     3.706760e+04
std      3.732740e+05
min      0.000000e+00
25%      2.363747e+02
50%      8.943452e+02
75%      3.637583e+03
max      8.277723e+06
Name: ibnr_dt, dtype: float64

ibnr_reel describe :
count    7.740000e+02
mean     2.440923e+04
std      2.815160e+05
min     -1.784000e+03
25%      8.250000e+00
50%      2.120000e+02
75%      1.553000e+03
max      6.696665e+06
Name: ibnr_reel, dtype: float64

Nombre de ibnr_reel <= 0 : 138
Nombre de ibnr_dt <= 0 : 1


In [ ]:
import sys
from pathlib import Path
project_root = Path.cwd().parent
sys.path.append(str(project_root))

%load_ext autoreload
%autoreload 2

from src.reserving.data import build_reserving_dataset, build_sequences
from src.reserving.deep_triangle import TriangleSeqDataset, DeepTriangleGRU, train_deep_triangle, predict_future_increments

df_full, observed, future = build_reserving_dataset()
X, M, keys, premiums = build_sequences(df_full, min_premium=1000)

dataset = TriangleSeqDataset(X, M)
model_dt = DeepTriangleGRU(hidden_dim=16)
optimizer = torch.optim.Adam(model_dt.parameters(), lr=1e-3)
model_dt = train_deep_triangle(model_dt, dataset, n_epochs=200, optimizer=optimizer, device=device)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Compagnies exclues (prime < 1,000$) : 51
Epoch 1/200 — loss: 0.013970
Epoch 21/200 — loss: 0.002114
Epoch 41/200 — loss: 0.001959
Epoch 61/200 — loss: 0.001826
Epoch 81/200 — loss: 0.001786
Epoch 101/200 — loss: 0.001755
Epoch 121/200 — loss: 0.001763
Epoch 141/200 — loss: 0.001805
Epoch 161/200 — loss: 0.001807
Epoch 181/200 — loss: 0.001850
Epoch 200/200 — loss: 0.001733


In [ ]:
results_dt = []
for i, (grcode, ay) in enumerate(keys):
    obs_len = int(M[i].sum())
    if obs_len == 0 or obs_len == 10:
        continue
    obs_seq = X[i, :obs_len]
    future_preds_scaled = predict_future_increments(model_dt, obs_seq, device)
    predicted_ibnr = future_preds_scaled.sum() * premiums[i]
    true_seq_future = df_full[
        (df_full["GRCODE"] == grcode) & (df_full["AccidentYear"] == ay) & (df_full["DevelopmentLag"] > obs_len)
    ]
    true_ibnr_val = true_seq_future["IncrementalPaid"].sum()
    results_dt.append({
        "grcode": grcode, "accident_year": ay, "obs_len": obs_len,
        "ibnr_dt": predicted_ibnr, "ibnr_reel": true_ibnr_val
    })

results_dt = pd.DataFrame(results_dt)
results_dt["ratio"] = results_dt["ibnr_dt"] / results_dt["ibnr_reel"]
clean = results_dt.replace([np.inf, -np.inf], np.nan).dropna(subset=["ratio"])

print("ibnr_dt describe (doit être positif maintenant) :")
print(results_dt["ibnr_dt"].describe())
print()
print("Ratio médian global :", clean["ratio"].median())
print("Ratio moyen tronqué (5-95%) :", clean["ratio"].clip(clean["ratio"].quantile(0.05), clean["ratio"].quantile(0.95)).mean())
print()
print("Par obs_len :")
print(clean.groupby("obs_len")["ratio"].median())

ibnr_dt describe (doit être positif maintenant) :
count    7.740000e+02
mean     2.156693e+04
std      2.707304e+05
min      0.000000e+00
25%      0.000000e+00
50%      1.207309e+02
75%      1.407214e+03
max      6.598400e+06
Name: ibnr_dt, dtype: float64

Ratio médian global : 0.6730486297060101
Ratio moyen tronqué (5-95%) : 0.837546207969106

Par obs_len :
obs_len
1    1.112117
2    1.124891
3    1.110260
4    0.934163
5    0.645522
6    0.140532
7    0.000000
8    0.000000
9   -0.000000
Name: ratio, dtype: float64


In [ ]:
model_dt = DeepTriangleGRU(hidden_dim=16)
optimizer = torch.optim.Adam(model_dt.parameters(), lr=1e-3)
model_dt = train_deep_triangle(model_dt, dataset, n_epochs=200, optimizer=optimizer, device=device)

results_dt = []
for i, (grcode, ay) in enumerate(keys):
    obs_len = int(M[i].sum())
    if obs_len == 0 or obs_len == 10:
        continue
    obs_seq = X[i, :obs_len]
    future_preds_scaled = predict_future_increments(model_dt, obs_seq, device)
    predicted_ibnr = future_preds_scaled.sum() * premiums[i]
    true_seq_future = df_full[
        (df_full["GRCODE"] == grcode) & (df_full["AccidentYear"] == ay) & (df_full["DevelopmentLag"] > obs_len)
    ]
    true_ibnr_val = true_seq_future["IncrementalPaid"].sum()
    results_dt.append({
        "grcode": grcode, "accident_year": ay, "obs_len": obs_len,
        "ibnr_dt": predicted_ibnr, "ibnr_reel": true_ibnr_val
    })

results_dt = pd.DataFrame(results_dt)
results_dt["ratio"] = results_dt["ibnr_dt"] / results_dt["ibnr_reel"]
clean = results_dt.replace([np.inf, -np.inf], np.nan).dropna(subset=["ratio"])

print("Ratio médian global :", clean["ratio"].median())
print("Ratio moyen tronqué (5-95%) :", clean["ratio"].clip(clean["ratio"].quantile(0.05), clean["ratio"].quantile(0.95)).mean())
print()
print("Par obs_len :")
print(clean.groupby("obs_len")["ratio"].median())
print()
print("Rappel — sans contrainte : 1.41 | avec Softplus : 2.75 | Mack : 1.06")

Epoch 1/200 — loss: 0.005737
Epoch 21/200 — loss: 0.001951
Epoch 41/200 — loss: 0.001805
Epoch 61/200 — loss: 0.001899
Epoch 81/200 — loss: 0.001813
Epoch 101/200 — loss: 0.001851
Epoch 121/200 — loss: 0.001759
Epoch 141/200 — loss: 0.001764
Epoch 161/200 — loss: 0.001766
Epoch 181/200 — loss: 0.001845
Epoch 200/200 — loss: 0.001735
Ratio médian global : 0.5697296679667805
Ratio moyen tronqué (5-95%) : 0.7022132828205376

Par obs_len :
obs_len
1    1.069680
2    1.085504
3    1.004548
4    0.820869
5    0.535402
6    0.063114
7    0.000000
8    0.000000
9    0.000000
Name: ratio, dtype: float64

Rappel — sans contrainte : 1.41 | avec Softplus : 2.75 | Mack : 1.06


In [ ]:
from src.reserving.models import split_conformal_calibration

test_conformal, q_hat = split_conformal_calibration(full_results, alpha=0.10)

print("Quantile calibré (q_hat) :", q_hat)
print("Taille du jeu de test :", len(test_conformal))
print()
print("Couverture empirique (conformal) :", test_conformal["covered_conformal"].mean())
print("Rappel — couverture Mack brute (sur tout le portefeuille) : 0.7438")

# Comparaison de la largeur moyenne des intervalles
width_conformal = (test_conformal["upper_conformal"] - test_conformal["lower_conformal"]).mean()
width_mack = (test_conformal["upper_90"] - test_conformal["lower_90"]).mean() if "upper_90" in test_conformal else None
print("\nLargeur moyenne intervalle conforme :", width_conformal)
if width_mack:
    print("Largeur moyenne intervalle Mack (référence) :", width_mack)
    

NameError: name 'full_results' is not defined

In [ ]:
torch.save(model_dt.state_dict(), "../models/deep_triangle.pt")